# LLM Agora Persona Demo
Interactively run a persona-driven Agora session that mirrors the CLI experience.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Adjust persona/question IDs and models to explore different matchups.
- The same helpers power the CLI (`agora persona ...`).

This walkthrough uses the bundled `prompts/default.yaml` template set (passed via `prompt_set='default'`).
Select a different YAML file in `src/agora/prompts/` by changing the prompt set name.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.append("../src")
prompt_set = 'default'
load_dotenv()

import matplotlib.pyplot as plt
from agora.plotting import collect_agent_metrics, plot_metrics
from agora.workflows import (
    build_persona_agent_configs,
    load_persona_catalog,
    load_question_catalog,
    print_agent_histories,
    run_debate_session,
)

In [ ]:
personas = load_persona_catalog('../data/personas.json')
questions = load_question_catalog('../data/questions.json')

# Persona debate configuration
Configure participant personas, question, and runtime controls.

In [ ]:
alpha_persona_id = 'high_wealth_founder'
beta_persona_id = 'unionized_warehouse_worker'
question_id = 'work'

alpha_model = 'openai/gpt-4o-mini'
beta_model = 'anthropic/claude-sonnet-4.5'

turns_per_agent = 5
snapshot_path = Path('snapshots/reflection_snapshot.json')
load_snapshot_flag = False
save_snapshot_flag = True
skip_first_agent_first_reflection = True

agent_configs = build_persona_agent_configs(
    alpha_persona_id=alpha_persona_id,
    beta_persona_id=beta_persona_id,
    question_id=question_id,
    personas=personas,
    questions=questions,
    alpha_model=alpha_model,
    beta_model=beta_model,
    prompt_set=prompt_set,
)

persona_agora, persona_agents = run_debate_session(
    agent_configs,
    turns_per_agent=turns_per_agent,
    verbose=True,
    snapshot_path=snapshot_path,
    load_snapshot_flag=load_snapshot_flag,
    save_snapshot_flag=save_snapshot_flag,
    skip_first_agent_first_reflection=skip_first_agent_first_reflection,
)

print_agent_histories(persona_agents)

# Metrics Visualization
Plot the evolution of public and off-record metrics across rounds.

In [ ]:
agent_metrics = collect_agent_metrics(persona_agora)

            agent_to_persona = {'Alpha': alpha_persona_id, 'Beta': beta_persona_id}
            persona_colors = {alpha_persona_id: '#2ecc71', beta_persona_id: '#e74c3c'}

            fig, axes = plot_metrics(
                agent_metrics,
                agent_to_persona=agent_to_persona,
                persona_colors=persona_colors
            )
            plt.show()

            print('
--- Metrics Summary ---')
            for agent_name, data in agent_metrics.items():
                display_name = agent_to_persona.get(agent_name, agent_name)
                print(f"{display_name}: {len(data['public'])} public, {len(data['off_record'])} off-record")